# 울산 태양광 발전량 예측 — 오렌지3와 같은 것을 코드로

오렌지에서 만든 워크플로를 파이썬 코드로 옮긴 것입니다.
**각 단계 제목에 대응하는 오렌지 위젯 이름을 적어 두었습니다.**

| 오렌지 위젯 | 이 노트북 |
|---|---|
| File | 1단계 |
| Data Table | 2단계 |
| Scatter Plot ×2 | 3단계 |
| Linear Regression · Random Forest · Neural Network + Test and Score | 5단계 |
| Predictions + Scatter Plot | 6단계 |
| Select Columns | 7단계 |

## 0단계 · 준비

한글 그래프를 위한 도구를 설치합니다. 한 번만 실행하면 됩니다.

In [ ]:
!pip install koreanize-matplotlib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib          # 그래프의 한글이 깨지지 않게 해준다

import warnings                      # 학습 중 뜨는 안내 문구를 숨긴다 (오류가 아니다)
warnings.filterwarnings("ignore")

print("준비 완료")

## 1단계 · 데이터 불러오기
### 🍊 오렌지의 `File` 위젯에 해당

실행하면 파일 선택 창이 뜹니다. `울산_태양광_오렌지용.csv` 를 고르세요.

In [ ]:
from google.colab import files

업로드 = files.upload()                    # 파일 선택 창을 띄운다
파일명 = list(업로드.keys())[0]             # 올린 파일의 이름을 꺼낸다

df = pd.read_csv(파일명)                   # 표 형태로 읽어들인다
print(파일명, "불러오기 완료")

## 2단계 · 데이터 확인
### 🍊 오렌지의 `Data Table` 위젯에 해당

오렌지에서 왼쪽 아래에 `8760 instances` 라고 떴던 그 숫자를 여기서 확인합니다.

In [ ]:
print("행 수:", len(df))                   # 8760 이어야 한다 (365일 x 24시간)
print("열 이름:", list(df.columns))
print()

df.head()                                  # 앞의 5줄만 보기

## 3단계 · 그림으로 먼저 보기
### 🍊 오렌지의 `Scatter Plot` 위젯 두 개에 해당

오렌지에서는 x축만 바꿔 두 개를 만들었습니다. 코드도 똑같습니다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# 첫 번째 - 하루 안에서의 변화 (오렌지: x=시)
ax[0].scatter(df["시"], df["발전량"], s=5, alpha=0.15)
ax[0].set_xlabel("시")
ax[0].set_ylabel("발전량 (MWh)")
ax[0].set_title("하루 안에서의 변화")

# 두 번째 - 1년 안에서의 변화 (오렌지: x=월)
ax[1].scatter(df["월"], df["발전량"], s=5, alpha=0.15, color="green")
ax[1].set_xlabel("월")
ax[1].set_ylabel("발전량 (MWh)")
ax[1].set_title("1년 안에서의 변화")

plt.tight_layout()
plt.show()

## 4단계 · 재료와 정답 정하기
### 🍊 오렌지 `File` 위젯에서 Role 을 바꾸던 그 작업

오렌지에서 클릭으로 했던 것을 코드로는 이렇게 씁니다.

| 오렌지의 Role | 여기서는 |
|---|---|
| target | `y` 에 담는다 |
| feature | `X` 에 담는다 |
| meta | 둘 다에 넣지 않는다 |

**`날짜` 를 재료에 넣지 않는 것이 중요합니다.** 넣으면 모델이 패턴을 배우는 대신 답을 통째로 외워 버립니다.

In [ ]:
재료 = ["월", "시", "요일"]                 # 오렌지의 feature
정답 = "발전량"                             # 오렌지의 target
                                           # 날짜는 meta 이므로 빼 둔다

X = pd.get_dummies(df[재료], columns=["요일"]).astype(float)
#   요일은 글자라서 계산할 수 없다. 요일마다 칸을 나눠 0과 1로 바꿔 준다.
#   오렌지는 이 작업을 자동으로 해 준다.

y = df[정답]

print("재료 칸 수:", X.shape[1])
print(list(X.columns))

## 5단계 · 모델 세 개를 한꺼번에 비교
### 🍊 오렌지의 `Linear Regression` · `Random Forest` · `Neural Network` + `Test and Score`

오렌지의 Test and Score 는 **교차검증(Cross validation)** 을 합니다.
데이터를 5조각으로 나눠, 4조각으로 배우고 남은 1조각을 맞히기를 5번 반복하는 방식입니다.

**설정값은 오렌지의 기본값과 똑같이 맞췄습니다.** 그래서 결과 숫자도 거의 같게 나옵니다.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

교차검증 = KFold(n_splits=5, shuffle=True, random_state=0)   # 5조각으로 나눈다

모델들 = {
    "Linear Regression": LinearRegression(),

    "Random Forest":     RandomForestRegressor(n_estimators=10,
                                               min_samples_split=5,
                                               random_state=0),

    "Neural Network":    make_pipeline(
                             StandardScaler(),                # 신경망은 숫자 크기를 맞춰 줘야 한다
                             MLPRegressor(hidden_layer_sizes=(100,),
                                          max_iter=200,
                                          random_state=42)),
}

결과 = []
예측값 = {}

for 이름, 모델 in 모델들.items():
    예측 = cross_val_predict(모델, X, y, cv=교차검증)          # 5번 반복해서 전체를 예측
    예측값[이름] = 예측
    결과.append({
        "Model": 이름,
        "MSE":  mean_squared_error(y, 예측),
        "RMSE": np.sqrt(mean_squared_error(y, 예측)),
        "MAE":  mean_absolute_error(y, 예측),
        "R2":   r2_score(y, 예측),
    })

pd.DataFrame(결과).round(3)

### 결과 읽는 법

**`MAE` 열만 보세요.** 숫자가 작을수록 잘 맞힌 것입니다.

- 오렌지에서도 랜덤포레스트가 1등이었습니다
- 신경망(딥러닝)이 2등입니다 — **더 복잡한 모델이 항상 이기는 것은 아닙니다**
- 선형회귀는 크게 뒤처집니다

## 6단계 · 예측이 맞았는지 눈으로 보기
### 🍊 오렌지의 `Predictions` + `Scatter Plot`

오렌지에서 `Axis x = 발전량`, `Axis y = Random Forest` 로 두었던 그 그림입니다.

In [ ]:
예측 = 예측값["Random Forest"]

plt.figure(figsize=(6.5, 6.5))
plt.scatter(y, 예측, s=6, alpha=0.12)

최대 = y.max()
plt.plot([0, 최대], [0, 최대], "r--", linewidth=1.5)   # 완벽하게 맞혔다면 점이 이 선 위에 놓인다

plt.xlabel("실제 발전량 (MWh)")
plt.ylabel("예측 발전량 (MWh)")
plt.title("실제 vs 예측  ·  Random Forest")
plt.tight_layout()
plt.show()

### 이 그림에서 볼 것

**빨간 대각선 위쪽은 점이 꽉 찼는데, 오른쪽 아래는 비어 있습니다.**

대각선 위 = 모델은 많이 나올 거라 했는데 실제로는 적게 나온 날입니다.
실제가 5인데 예측이 30인 점을 찾아보세요.

> **그날 무슨 일이 있었을까요?**

## 7단계 · 변수를 바꿔보는 실험
### 🍊 오렌지의 `Select Columns` 위젯

오렌지에서 변수를 체크했다 풀었다 하던 작업입니다.

In [ ]:
실험목록 = [
    ["월", "시", "요일"],      # 전부
    ["월", "시"],              # 요일 빼기
    ["시"],                    # 시각만
    ["월"],                    # 월만
]

기록 = []
for 재료 in 실험목록:
    Xi = pd.get_dummies(df[재료], columns=[c for c in 재료 if c == "요일"]).astype(float)
    모델 = RandomForestRegressor(n_estimators=10, min_samples_split=5, random_state=0)
    예측 = cross_val_predict(모델, Xi, y, cv=교차검증)
    기록.append({"사용한 재료": " + ".join(재료),
                 "MAE": round(mean_absolute_error(y, 예측), 3)})

pd.DataFrame(기록)

### 실험 결과에서 볼 것

**`요일` 을 빼면 오히려 더 잘 맞힙니다.** (3.755 → 3.349)

태양광 발전량은 무슨 요일인지와 아무 상관이 없기 때문입니다.
해는 월요일이라고 더 뜨지 않습니다.
쓸모없는 재료는 도움이 안 되는 정도가 아니라 **방해가 됩니다.**

`시` 하나만 쓰면 4.255, `월` 하나만 쓰면 12.313 입니다.
시각이 월보다 훨씬 중요한 재료라는 뜻입니다.

> 변수는 **많이** 넣는 것이 아니라 **맞는 것**을 넣는 것입니다.

## 오늘 남은 질문

이 모델은 6월 10일과 6월 11일에 **똑같은 값**을 예측합니다.
재료가 `월` 과 `시` 뿐이라, 모델에게 두 날은 완전히 같은 날이기 때문입니다.

두 날의 실제 발전량이 달랐다면, 무엇이 달랐기 때문일까요?

> **모델이 못 맞힌 것이 아니라, 우리가 알려주지 않은 것입니다.**